# 14 · Reproducibilidad y semillas aleatorias

Las simulaciones estocásticas requieren controlar el generador pseudoaleatorio para poder reproducir experimentos computacionales y comparar cambios en el código.

> **ProteinMotorSim 0.1.1** · Ejemplo educativo. Las tasas predeterminadas del modelo son ilustrativas y no representan todavía una parametrización experimental validada.

## Preparación del entorno

Esta celda localiza automáticamente la raíz del repositorio e incorpora `src/` a la ruta de importación. Instala antes las dependencias con `python -m pip install -e ".[dev,viz]"` y selecciona el kernel `Python (ProteinMotorSim)`.

In [1]:
from pathlib import Path
import sys

# Busca la raíz del proyecto desde el directorio actual o sus carpetas padre.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in _candidates if (path / "pyproject.toml").exists() and (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No se encontró ProteinMotorSim. Abre JupyterLab desde la carpeta raíz "
        "del proyecto o desde su carpeta notebooks/."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Proyecto detectado: {PROJECT_ROOT}")

Proyecto detectado: /home/zix/ProteinMotorSim


In [2]:
from protein_motor_sim import Environment, GillespieSimulator, Kinesin1

simulator = GillespieSimulator()
motor = Kinesin1()
environment = Environment(atp_concentration=1.0e-3)

def run_with_seed(seed: int):
    """Ejecuta una trayectoria reproducible con una semilla dada."""
    return simulator.run(motor, environment, t_max=0.25, seed=seed)

a = run_with_seed(2026)
b = run_with_seed(2026)
c = run_with_seed(2027)

## 1. Comparar dos ejecuciones con la misma semilla

In [3]:
print("Mismos tiempos:", a.times_s == b.times_s)
print("Mismas posiciones:", a.positions_nm == b.positions_nm)
print("Mismos eventos:", a.events == b.events)

Mismos tiempos: True
Mismas posiciones: True
Mismos eventos: True


## 2. Comparar ejecuciones con semillas diferentes

In [4]:
print("Semilla 2026: eventos =", len(a.events), "desplazamiento =", a.displacement_nm, "nm")
print("Semilla 2027: eventos =", len(c.events), "desplazamiento =", c.displacement_nm, "nm")
print("Trayectorias idénticas:", a.events == c.events and a.times_s == c.times_s)

Semilla 2026: eventos = 40 desplazamiento = 98.40000000000002 nm
Semilla 2027: eventos = 39 desplazamiento = 98.40000000000002 nm
Trayectorias idénticas: False


## 3. Buenas prácticas

Registra siempre la semilla, los parámetros, las condiciones experimentales, la versión del código y las dependencias. La reproducibilidad exacta puede depender también de las versiones de NumPy y del entorno de ejecución.